In [55]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Load Data 

In [77]:
data_path = Path("../data")

movie_path = f"{data_path}/refine_movies.csv"
rating_path = f"{data_path}/refine_ratings.csv"
tag_path = f"{data_path}/refine_tags.csv"
link_path = f"{data_path}/refine_links.csv"

In [78]:
movies_df = pd.read_csv(
    movie_path,
    usecols=['movieId', 'title','year','genres'],
    dtype={'title':str, 'year':str, 'movieId':int, 'genres':str})

ratings_df = pd.read_csv(
    rating_path,
    usecols=['userId','movieId','rating'],
    dtype={'userId':int, 'movieId':int, 'rating':float}
    )
tags_df = pd.read_csv(
    tag_path,
    usecols=['userId','movieId','tag'],
    dtype={'userId':int, 'movieId':int, 'tag':str})
links_df = pd.read_csv(link_path)

In [79]:
df_list = {
    "movies" : movies_df,
    "ratings" : ratings_df,
    "tags" : tags_df,
    "links_df" : links_df
}

# Data check

In [80]:
# genres to list
import ast
movies_df['genres'] = movies_df['genres'].apply(lambda x : ast.literal_eval(x))

movies_df['genres']

0       [Adventure, Animation, Children, Comedy, Fantasy]
1                          [Adventure, Children, Fantasy]
2                                       [Comedy, Romance]
3                                [Comedy, Drama, Romance]
4                                                [Comedy]
                              ...                        
9734                 [Action, Animation, Comedy, Fantasy]
9735                         [Animation, Comedy, Fantasy]
9736                                              [Drama]
9737                                  [Action, Animation]
9738                                             [Comedy]
Name: genres, Length: 9739, dtype: object

In [81]:
ratings_df

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0
...,...,...,...
100831,610,166534,4.0
100832,610,168248,5.0
100833,610,168250,5.0
100834,610,168252,5.0


In [82]:
tags_df

,userId,movieId,tag
0,2,60756,['funny']
1,2,60756,"['Highly', 'quotable']"
2,2,60756,"['will', 'ferrell']"
3,2,89774,"['Boxing', 'story']"
4,2,89774,['MMA']
...,...,...,...
3678,606,7382,"['for', 'katie']"
3679,606,7936,['austere']
3680,610,3265,"['gun', 'fu']"
3681,610,3265,"['heroic', 'bloodshed']"


# (1) STANDARDIZATION

In [83]:
temp_rating = ratings_df.copy()

func = lambda rating : (rating - rating.mean()) / rating.std()

temp_rating['centered_rating'] = ratings_df.groupby(by='userId')['rating'].transform(func) # apply 는 원래 구조를 변경, transform 는 원래 DF 구조를 유지

temp_rating['centered_rating'] = temp_rating['centered_rating'].fillna(0) # 평가가 1개인 경우 std=0 이라서 NaN 값이 출력 → 0 으로 대체

ratings_df = temp_rating

In [84]:
movie_ratings_df = pd.merge(ratings_df, movies_df, on='movieId', how='left') # on : merge key
movie_ratings_df

,userId,movieId,rating,centered_rating,title,genres,year
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995)
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",(1995)
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",(1995)
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",(1995)
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",(1995)
...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",(2017)
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",(2017)
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],(2017)
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",(2017)


# zero ratings movie check

In [85]:
movie_ratings_df.isnull().sum()

userId             0
movieId            0
rating             0
centered_rating    0
title              0
genres             0
year               0
dtype: int64

In [86]:
no_ratings_movieId = movie_ratings_df.loc[movie_ratings_df.isnull().any(axis=1)]['movieId'].values
print("평가가 존재하지 않는 영화 ID 갯수: ",len(no_ratings_movieId))
no_ratings_movieId

평가가 존재하지 않는 영화 ID 갯수:  0


array([], dtype=int64)

In [87]:
movie_ratings_df[movie_ratings_df['movieId'].isin(no_ratings_movieId)]

,userId,movieId,rating,centered_rating,title,genres,year


In [88]:
# double check (ratings_df 에도 존재하지 않는지 확인)
ratings_df[ratings_df['movieId'].isin(no_ratings_movieId)]

,userId,movieId,rating,centered_rating


# (2) 유저별/영화별 평균평점

In [89]:
user_rating_mean = ratings_df.groupby(by='userId')['rating'].apply(lambda x: x.mean())
user_rating_mean

userId
1      4.366379
2      3.948276
3      2.435897
4      3.555556
5      3.636364
         ...   
606    3.657399
607    3.786096
608    3.134176
609    3.270270
610    3.688556
Name: rating, Length: 610, dtype: float64

In [90]:
movie_rating_mean = ratings_df.groupby(by='movieId')['rating'].apply(lambda x: x.mean())
movie_rating_mean


movieId
1         3.920930
2         3.431818
3         3.259615
4         2.357143
5         3.071429
            ...   
193581    4.000000
193583    3.500000
193585    3.500000
193587    3.500000
193609    4.000000
Name: rating, Length: 9721, dtype: float64

# 3차발표 개선사항 추가 - mean 과 median 비교

In [91]:
# 3 차 발표 추가 내용
user_rating_median = ratings_df.groupby(by='userId')['rating'].apply(lambda x: x.median())
movie_rating_median = ratings_df.groupby(by='movieId')['rating'].apply(lambda x: x.median())

In [92]:
temp_df = movies_df.set_index('movieId')
temp_df = temp_df.join(movie_rating_mean.rename("avg_rating"))
temp_df = temp_df.join(movie_rating_median.rename("median_rating")) # 추가내용
movies_df = temp_df.reset_index()

In [93]:
movies_df # movies_df 에 각 영화별 평균 평점 반영

,movieId,title,genres,year,avg_rating,median_rating
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),3.920930,4.0
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",(1995),3.431818,3.5
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",(1995),3.259615,3.0
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",(1995),2.357143,3.0
4,5,Father of the Bride Part II (1995),[Comedy],(1995),3.071429,3.0
...,...,...,...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",(2017),4.000000,4.0
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",(2017),3.500000,3.5
9736,193585,Flint (2017),[Drama],(2017),3.500000,3.5
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",(2018),3.500000,3.5


## 추가 작업 - 영화당 평균 평점 갯수 계산
- a. 영화 평점 갯수 1개, 5.0
  
- b. 영화 평점 갯수 100개, 4.5

- 영화 평점 갯수가 적은 영화에 대해 평균 평점 조정

In [94]:
item_user_matrix = movie_ratings_df.pivot_table(values='rating', index='movieId', columns='userId')
print("평균 평점 갯수 :" ,round(item_user_matrix.apply(lambda x: x.count(), axis=1).mean()))
ratings_count = item_user_matrix.apply(lambda x: x.count(), axis=1)
ratings_count

평균 평점 갯수 : 10


movieId
1         215
2         110
3          52
4           7
5          49
         ... 
193581      1
193583      1
193585      1
193587      1
193609      1
Length: 9721, dtype: int64

In [95]:
ratings_count.sort_values(ascending=False)

movieId
356       329
318       317
296       307
593       279
2571      278
         ... 
58351       1
4089        1
58492       1
4083        1
193609      1
Length: 9721, dtype: int64

In [96]:
threshold = ratings_count.sort_values().quantile(0.9) # 상위 20% 에 해당하는 평점갯수를 threshold 설정
threshold

27.0

In [97]:
temp_df = movies_df.set_index(keys='movieId').copy()
temp_df = temp_df.join(ratings_count.rename('count'))
movies_df = temp_df.reset_index() # movies_df 에 평점갯수도 반영

In [98]:
movies_df = movies_df.dropna(subset='count')
movies_df['count'] = movies_df['count'].astype('int64')

movies_df

,movieId,title,genres,year,avg_rating,median_rating,count
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),3.920930,4.0,215
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",(1995),3.431818,3.5,110
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",(1995),3.259615,3.0,52
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",(1995),2.357143,3.0,7
4,5,Father of the Bride Part II (1995),[Comedy],(1995),3.071429,3.0,49
...,...,...,...,...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",(2017),4.000000,4.0,1
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",(2017),3.500000,3.5,1
9736,193585,Flint (2017),[Drama],(2017),3.500000,3.5,1
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",(2018),3.500000,3.5,1


In [99]:
nrating_weighted_movieid = movies_df[movies_df['count'] > threshold]['movieId'] # 평점 갯수가 상위 20% 영화
print("상향 조정할 영화 갯수 : ",len(nrating_weighted_movieid))

상향 조정할 영화 갯수 :  933


In [103]:
movies_mean = movies_df['avg_rating'].mean() # 전체 영화 평균

movies_median = movies_df['avg_rating'].median() # 추가내용

print("전체 영화 평균 점수 : ",movies_mean)
print("전체 영화 중앙값 점수 : ",movies_median) # 추가내용
def weighted_rating(df, threshold=threshold, mean=movies_mean):
    c = df['count']
    r = df['avg_rating']
    return ( c / (c+threshold)* r ) + ( threshold / (c +threshold) * mean)

movies_df['weighted_avg_rating'] = movies_df.apply(weighted_rating, axis=1) # 리뷰가 많은 영화 → avg_rating 을 높여준다.

movies_df['weighted_median_rating'] = movies_df.apply(lambda x: weighted_rating(x, mean=movies_median), axis=1) # 추가내용
movies_df

전체 영화 평균 점수 :  3.2623878226789222
전체 영화 중앙값 점수 :  3.4166666666666665


,movieId,title,genres,year,avg_rating,median_rating,count,weighted_avg_rating,weighted_median_rating
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),3.920930,4.0,215,3.847456,3.864669
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",(1995),3.431818,3.5,110,3.398427,3.428832
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",(1995),3.259615,3.0,52,3.260563,3.313291
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",(1995),2.357143,3.0,7,3.076014,3.198529
4,5,Father of the Bride Part II (1995),[Comedy],(1995),3.071429,3.0,49,3.139269,3.194079
...,...,...,...,...,...,...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",(2017),4.000000,4.0,1,3.288731,3.437500
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",(2017),3.500000,3.5,1,3.270874,3.419643
9736,193585,Flint (2017),[Drama],(2017),3.500000,3.5,1,3.270874,3.419643
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",(2018),3.500000,3.5,1,3.270874,3.419643


# $ \frac{count}{count+threshold}* \text{avg} + \frac{threshold}{count+threshold}*mean$

- 각 영화별 평균점수 = avg
- 전체 영화의 평균점수 = mean

count 가 낮은 영화에 대해서 mean 보다 평점이 높다면, 평균 방향으로 값을 땡기게끔

In [104]:
temp_map = movies_df.set_index('movieId')['weighted_avg_rating']
temp_map2 = movies_df.set_index('movieId')['weighted_median_rating']

movie_ratings_df['weighted_rating'] = movie_ratings_df['rating'] - movie_ratings_df['movieId'].map(temp_map) # 각 사용자의 영화 점수 - 영화평균
movie_ratings_df['weighted_rating2'] = movie_ratings_df['rating'] - movie_ratings_df['movieId'].map(temp_map2) # 각 사용자의 영화 점수 - 영화평균

movie_ratings_df['weighted_centered_rating'] = movie_ratings_df[['userId','weighted_rating']].groupby(by='userId').transform(func) # z-score
movie_ratings_df['weighted_centered_rating2'] = movie_ratings_df[['userId','weighted_rating2']].groupby(by='userId').transform(func) # z-score

movie_ratings_df

,userId,movieId,rating,centered_rating,title,genres,year,weighted_rating,weighted_rating2,weighted_centered_rating,weighted_centered_rating2
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),0.152544,0.135331,-0.976645,-0.925151
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",(1995),0.739437,0.686709,-0.191300,-0.181726
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",(1995),0.197020,0.164729,-0.917130,-0.885513
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",(1995),1.108328,1.090217,0.302328,0.362326
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",(1995),0.876258,0.858225,-0.008215,0.049530
...,...,...,...,...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",(2017),0.724713,0.598485,0.558390,0.514684
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",(2017),1.556339,1.433824,1.684793,1.635545
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],(2017),1.605132,1.505952,1.750881,1.732328
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",(2017),1.248376,1.168269,1.267669,1.279223


In [26]:
weighted_item_user_matrix = movie_ratings_df.pivot_table(values='weighted_centered_rating', index='movieId', columns='userId')
weighted_item_user_matrix

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,-0.976645,NaN,NaN,NaN,0.119428,NaN,0.756712,NaN,NaN,NaN,...,-1.259866,NaN,0.043786,-1.382137,0.387122,-2.479115,-0.187531,-1.176557,-1.537489,1.137869
2,NaN,NaN,NaN,NaN,NaN,0.499628,NaN,0.624459,NaN,NaN,...,NaN,0.730678,NaN,1.966805,0.319085,NaN,NaN,-1.228964,NaN,NaN
3,-0.191300,NaN,NaN,NaN,NaN,1.917122,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.087213,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,-0.344476,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,2.068224,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-0.413722,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193585,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# (3) 유저별 장르 평균 점수

In [105]:
genres_list = movies_df['genres'].explode().unique()
genres_list

array(['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy',
       'Romance', 'Drama', 'Action', 'Crime', 'Thriller', 'Horror',
       'Mystery', 'Sci-Fi', 'War', 'Musical', 'Documentary', 'IMAX',
       'Western', 'Film-Noir'], dtype=object)

In [106]:
movies_df 

,movieId,title,genres,year,avg_rating,median_rating,count,weighted_avg_rating,weighted_median_rating
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),3.920930,4.0,215,3.847456,3.864669
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",(1995),3.431818,3.5,110,3.398427,3.428832
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",(1995),3.259615,3.0,52,3.260563,3.313291
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",(1995),2.357143,3.0,7,3.076014,3.198529
4,5,Father of the Bride Part II (1995),[Comedy],(1995),3.071429,3.0,49,3.139269,3.194079
...,...,...,...,...,...,...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",(2017),4.000000,4.0,1,3.288731,3.437500
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",(2017),3.500000,3.5,1,3.270874,3.419643
9736,193585,Flint (2017),[Drama],(2017),3.500000,3.5,1,3.270874,3.419643
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",(2018),3.500000,3.5,1,3.270874,3.419643


In [107]:
def calc_user_genre_mean(group):
    genres_totals = {genre : 0 for genre in genres_list}
    genres_count = {genre : 0 for genre in genres_list}
    
    for _, row in group.iterrows():
        rating = row['centered_rating']
        movie_genres = row['genres']
        for genre in movie_genres:
            genres_totals[genre] += rating
            genres_count[genre] += 1
    genres_avg = {}
    for genre in genres_list:
        if genres_count[genre] > 0 :
            genres_avg[genre] = genres_totals[genre]/genres_count[genre]
        else:
            genres_avg[genre] = pd.NA
    return pd.Series(genres_avg)

In [ ]:
def improved_calc_genre_mean(movies_ratings):
    temp_df = movies_ratings.explode('genres')
    genres_means = temp_df.groupby(['userId','genres'])['centered_rating'].mean()
    user_genre_avg = genres_means.unstack()
    return user_genre_avg

def improved_calc_genre_median(movies_ratings): # 추가 내용
    temp_df = movies_ratings.explode('genres')
    genres_means = temp_df.groupby(['userId','genres'])['centered_rating'].median()
    user_genre_avg = genres_means.unstack()
    return user_genre_avg

In [109]:
movie_ratings_df

,userId,movieId,rating,centered_rating,title,genres,year,weighted_rating,weighted_rating2,weighted_centered_rating,weighted_centered_rating2
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),0.152544,0.135331,-0.976645,-0.925151
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",(1995),0.739437,0.686709,-0.191300,-0.181726
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",(1995),0.197020,0.164729,-0.917130,-0.885513
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",(1995),1.108328,1.090217,0.302328,0.362326
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",(1995),0.876258,0.858225,-0.008215,0.049530
...,...,...,...,...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",(2017),0.724713,0.598485,0.558390,0.514684
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",(2017),1.556339,1.433824,1.684793,1.635545
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],(2017),1.605132,1.505952,1.750881,1.732328
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",(2017),1.248376,1.168269,1.267669,1.279223


In [110]:
user_genre_avg = movie_ratings_df.groupby('userId').apply(improved_calc_genre_mean)
user_genre_median = movie_ratings_df.groupby('userId').apply(improved_calc_genre_median)

/var/folders/1t/sts9dw9x317fjpv6tzhvwldw0000gn/T/ipykernel_29057/402604189.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_genre_avg = movie_ratings_df.groupby('userId').apply(improved_calc_genre_mean)
/var/folders/1t/sts9dw9x317fjpv6tzhvwldw0000gn/T/ipykernel_29057/402604189.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_genre_median = movie_ratings_df.groupby('userId').apply(improved_ca

In [111]:
user_genre_avg

,genres,Action,Adventure,Animation,Children,Comedy,Crime,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,Documentary,IMAX
userId,userId,,,,,,,,,,,,,,,,,,,
1,1,-0.055193,0.027318,0.404071,0.226536,-0.111582,-0.013529,0.203778,-0.085629,0.791978,-1.119672,0.394275,-0.249626,-0.073354,-0.176714,-0.276139,0.167016,-0.100825,NaN,NaN
2,2,0.007782,0.271086,NaN,NaN,0.064205,-0.184053,-0.081829,NaN,NaN,-1.177084,NaN,0.064205,0.684849,-0.090956,-0.308182,0.684849,-0.556440,0.477967,-0.246118
3,3,0.543150,0.030662,-0.925982,-0.925982,-0.686821,-0.925982,-0.806402,0.449193,NaN,1.076991,-0.925982,1.226467,-0.925982,0.843809,0.816476,-0.925982,NaN,NaN,NaN
4,4,-0.179238,0.075800,0.338185,0.186002,-0.034957,0.197275,-0.054955,0.097896,0.338185,0.528415,0.338185,-0.058815,-0.134108,-0.549551,-0.002225,0.012078,0.186002,0.338185,-0.422732
5,5,-0.530322,-0.390093,0.703697,0.479330,-0.171335,0.198871,0.165216,0.511382,NaN,-0.642506,0.771007,0.367146,-0.550719,-1.147331,-0.081588,-0.305955,-0.642506,NaN,0.030596
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,606,-0.660928,-0.212669,0.078560,-0.287824,-0.127159,-0.004507,0.180310,-0.082115,0.214192,-0.429825,0.096494,0.184789,0.114303,-0.138702,-0.182668,0.186307,-0.339218,0.196930,-0.821547
607,607,-0.066146,-0.330790,-0.468865,-0.378026,-0.475141,0.029740,0.234140,-0.222302,NaN,0.339861,-0.192715,0.891582,-0.278416,-0.555162,0.340346,0.394105,0.221511,NaN,1.257075
608,608,0.181744,0.080443,-0.014819,-0.624453,-0.368359,0.443672,0.281048,-0.124322,0.570598,0.171795,-0.348942,0.385957,-0.229215,0.150317,0.372944,0.412107,-0.461252,-0.124322,0.802237


In [ ]:
user_genre_median # 추가 (z-score 기준 median)

,genres,Action,Adventure,Animation,Children,Comedy,Crime,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,Documentary,IMAX
userId,userId,,,,,,,,,,,,,,,,,,,
1,1,-0.457947,0.791978,0.791978,0.791978,-0.457947,0.791978,0.791978,-0.457947,0.791978,-0.457947,0.791978,0.791978,-0.457947,-0.457947,-0.457947,0.791978,-0.457947,NaN,NaN
2,2,0.064205,0.064205,NaN,NaN,0.064205,0.064205,0.064205,NaN,NaN,-1.177084,NaN,0.064205,0.684849,-0.246118,0.064205,0.684849,-0.556440,1.305493,-0.246118
3,3,0.987306,0.030662,-0.925982,-0.925982,-0.925982,-0.925982,-0.925982,0.748145,NaN,1.106886,-0.925982,1.226467,-0.925982,1.226467,1.226467,-0.925982,NaN,NaN,NaN
4,4,0.338185,0.338185,0.338185,0.338185,0.338185,0.338185,0.338185,0.338185,0.718644,0.338185,0.338185,0.338185,-0.042273,-0.803190,0.338185,0.338185,0.338185,0.338185,-0.422732
5,5,-0.642506,-0.642506,0.871972,0.367146,-0.642506,0.367146,0.367146,0.367146,NaN,-0.642506,1.376798,0.367146,-0.642506,-1.147331,0.367146,0.367146,-0.642506,NaN,-0.642506
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,606,-0.217366,-0.217366,0.473127,-0.217366,0.473127,0.473127,0.473127,-0.217366,0.473127,-0.217366,0.473127,0.473127,0.473127,-0.217366,-0.217366,0.473127,-0.217366,0.473127,-0.907859
607,607,0.221511,-0.814053,-0.814053,-0.814053,-0.814053,0.221511,0.221511,-0.814053,NaN,0.221511,-0.814053,1.257075,-0.814053,-0.814053,0.221511,1.257075,0.221511,NaN,1.257075
608,608,0.338958,0.338958,0.338958,-0.587601,-0.124322,0.802237,0.338958,-0.124322,0.802237,0.338958,-0.124322,0.338958,-0.124322,0.338958,0.338958,0.802237,-0.124322,0.107318,0.802237


# 영화별 장르 매트릭스 계산

In [113]:
genres_list

array(['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy',
       'Romance', 'Drama', 'Action', 'Crime', 'Thriller', 'Horror',
       'Mystery', 'Sci-Fi', 'War', 'Musical', 'Documentary', 'IMAX',
       'Western', 'Film-Noir'], dtype=object)

In [114]:
movies_df

,movieId,title,genres,year,avg_rating,median_rating,count,weighted_avg_rating,weighted_median_rating
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),3.920930,4.0,215,3.847456,3.864669
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",(1995),3.431818,3.5,110,3.398427,3.428832
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",(1995),3.259615,3.0,52,3.260563,3.313291
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",(1995),2.357143,3.0,7,3.076014,3.198529
4,5,Father of the Bride Part II (1995),[Comedy],(1995),3.071429,3.0,49,3.139269,3.194079
...,...,...,...,...,...,...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",(2017),4.000000,4.0,1,3.288731,3.437500
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",(2017),3.500000,3.5,1,3.270874,3.419643
9736,193585,Flint (2017),[Drama],(2017),3.500000,3.5,1,3.270874,3.419643
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",(2018),3.500000,3.5,1,3.270874,3.419643


In [115]:
movie_genre_matrix = pd.DataFrame(0, index=movies_df['movieId'], columns=genres_list)
movie_genre_matrix

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir
movieId,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
193583,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
193585,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [116]:
for genre in genres_list:
    movie_genre_matrix[genre] = movies_df.set_index('movieId')['genres'].apply(lambda x : 1 if genre in x else 0) # apply 적용을 위해 movies_df 의 인덱스 movieId 로 설정
    
movie_genre_matrix

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir
movieId,,,,,,,,,,,,,,,,,,,
1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,0,1,0,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0
193583,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
193585,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0


In [117]:
genre_count = movie_genre_matrix.sum(axis=1)

movie_genre_matrix = movie_genre_matrix.div(genre_count.replace(0,1), axis=0) # 장르가 많은 영화의 점수가 많이 반영되지 않도록 정규화

movie_genre_matrix

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir
movieId,,,,,,,,,,,,,,,,,,,
1,0.200000,0.200000,0.200000,0.200000,0.200000,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.333333,0.000000,0.333333,0.000000,0.333333,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.000000,0.500000,0.000000,0.500000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.000000,0.333333,0.000000,0.333333,0.333333,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,0.000000,0.250000,0.000000,0.250000,0.250000,0.000000,0.000000,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
193583,0.000000,0.333333,0.000000,0.333333,0.333333,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
193585,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [118]:
print(user_genre_avg.shape)
print(movie_genre_matrix.shape)

(610, 19)
(9721, 19)


# (4) Year

In [119]:
movies_year = movies_df['year'].str.strip(to_strip="()")
movies_year

0       1995
1       1995
2       1995
3       1995
4       1995
        ... 
9734    2017
9735    2017
9736    2017
9737    2018
9738    1991
Name: year, Length: 9721, dtype: object

## 연도 추가 전처리

In [120]:
movies_year[movies_year.apply(lambda x : len(x) > 4)]

9515    2006–2007
Name: year, dtype: object

In [121]:
movies_year[movies_year.apply(lambda x : len(x) > 4)]
movies_df.loc[9515, 'year'] = "(2006)"
movies_df.loc[9515]

movieId                                                  171749
title                         Death Note: Desu nôto (2006–2007)
genres                    [Animation, Mystery, Sci-Fi, Fantasy]
year                                                     (2006)
avg_rating                                                  5.0
median_rating                                               5.0
count                                                         1
weighted_avg_rating                                    3.324445
weighted_median_rating                                 3.473214
Name: 9515, dtype: object

In [122]:
movies_df

,movieId,title,genres,year,avg_rating,median_rating,count,weighted_avg_rating,weighted_median_rating
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),3.920930,4.0,215,3.847456,3.864669
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",(1995),3.431818,3.5,110,3.398427,3.428832
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",(1995),3.259615,3.0,52,3.260563,3.313291
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",(1995),2.357143,3.0,7,3.076014,3.198529
4,5,Father of the Bride Part II (1995),[Comedy],(1995),3.071429,3.0,49,3.139269,3.194079
...,...,...,...,...,...,...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",(2017),4.000000,4.0,1,3.288731,3.437500
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",(2017),3.500000,3.5,1,3.270874,3.419643
9736,193585,Flint (2017),[Drama],(2017),3.500000,3.5,1,3.270874,3.419643
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",(2018),3.500000,3.5,1,3.270874,3.419643


## 연도의 괄호 제거

In [123]:
movies_year = movies_df.set_index('movieId')['year'].str.strip(to_strip="()")

In [124]:
movies_year

movieId
1         1995
2         1995
3         1995
4         1995
5         1995
          ... 
193581    2017
193583    2017
193585    2017
193587    2018
193609    1991
Name: year, Length: 9721, dtype: object

In [125]:
movie_ratings_df['year'] = movie_ratings_df['movieId'].map(movies_year.to_dict())

movie_ratings_df

,userId,movieId,rating,centered_rating,title,genres,year,weighted_rating,weighted_rating2,weighted_centered_rating,weighted_centered_rating2
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995,0.152544,0.135331,-0.976645,-0.925151
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",1995,0.739437,0.686709,-0.191300,-0.181726
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",1995,0.197020,0.164729,-0.917130,-0.885513
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",1995,1.108328,1.090217,0.302328,0.362326
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",1995,0.876258,0.858225,-0.008215,0.049530
...,...,...,...,...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",2017,0.724713,0.598485,0.558390,0.514684
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",2017,1.556339,1.433824,1.684793,1.635545
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],2017,1.605132,1.505952,1.750881,1.732328
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",2017,1.248376,1.168269,1.267669,1.279223


### 연도가 조사되지 않은 영화 확인
- 전처리 체크

In [126]:
print("연도 영화하지 않는 영화 : " ,movie_ratings_df['year'].isnull().sum(), ", 전체 :", len(movie_ratings_df))

연도 영화하지 않는 영화 :  0 , 전체 : 100836


# 코드 정리

### 1. 데이터 load

### 2. movie-ratings df

ratings 과 movies 를 merge

### 3. item-user matrix

movie-ratings df 에 pivot 적용

### 4. movie-xxx matrix 계산

### 5.  dot 연산을 통한 movie-movie matrix

### 6. cos sim 계산해 영화 추천

In [127]:
movie_ratings_df

,userId,movieId,rating,centered_rating,title,genres,year,weighted_rating,weighted_rating2,weighted_centered_rating,weighted_centered_rating2
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995,0.152544,0.135331,-0.976645,-0.925151
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",1995,0.739437,0.686709,-0.191300,-0.181726
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",1995,0.197020,0.164729,-0.917130,-0.885513
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",1995,1.108328,1.090217,0.302328,0.362326
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",1995,0.876258,0.858225,-0.008215,0.049530
...,...,...,...,...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",2017,0.724713,0.598485,0.558390,0.514684
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",2017,1.556339,1.433824,1.684793,1.635545
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],2017,1.605132,1.505952,1.750881,1.732328
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",2017,1.248376,1.168269,1.267669,1.279223


## eval 에 필요한 data
x_test = ratings_df.copy() # movie_ratings_df(expanded version of ratings_df) → x_test

y_test = ratings_df['userId']

sim_matrix

user_rating_mean

item_user_matrix

In [128]:
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

In [129]:
movie_ratings_df

,userId,movieId,rating,centered_rating,title,genres,year,weighted_rating,weighted_rating2,weighted_centered_rating,weighted_centered_rating2
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995,0.152544,0.135331,-0.976645,-0.925151
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",1995,0.739437,0.686709,-0.191300,-0.181726
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",1995,0.197020,0.164729,-0.917130,-0.885513
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",1995,1.108328,1.090217,0.302328,0.362326
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",1995,0.876258,0.858225,-0.008215,0.049530
...,...,...,...,...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",2017,0.724713,0.598485,0.558390,0.514684
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",2017,1.556339,1.433824,1.684793,1.635545
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],2017,1.605132,1.505952,1.750881,1.732328
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",2017,1.248376,1.168269,1.267669,1.279223


In [130]:
x_train, x_test, y_train, y_test = train_test_split(movie_ratings_df, movie_ratings_df['userId'], stratify=movie_ratings_df['userId'], test_size=0.2, random_state=42)

In [131]:
movieid_idx = x_train.pivot_table(values="centered_rating", index='movieId', columns='userId').index # x_train 에서 사용한 movieid 추출

### MODEL EVAL

In [132]:
x_train

,userId,movieId,rating,centered_rating,title,genres,year,weighted_rating,weighted_rating2,weighted_centered_rating,weighted_centered_rating2
21452,140,4234,3.0,-0.618215,"Tailor of Panama, The (2001)","[Drama, Thriller]",2001,-0.252640,-0.382812,-0.419286,-0.481669
22899,156,2080,1.0,-2.928387,Lady and the Tramp (1955),"[Animation, Children, Comedy, Romance]",1955,-2.354689,-2.405488,-2.974350,-2.973550
58090,380,182639,4.0,0.340023,The Second Renaissance Part II (2003),"[Animation, Sci-Fi]",2003,0.711269,0.562500,0.444764,0.367027
79604,495,5254,3.5,-0.444205,Blade II (2002),"[Action, Horror, Thriller]",2002,0.269107,0.200820,-0.277789,-0.282044
100382,610,69134,3.0,-0.803054,Antichrist (2009),"[Drama, Fantasy]",2009,-0.346390,-0.476562,-0.892375,-0.927818
...,...,...,...,...,...,...,...,...,...,...,...
49818,318,136024,3.0,-1.388122,The Professional: Golgo 13 (1983),"[Action, Animation, Crime]",1983,-0.253017,-0.401786,-1.057528,-1.190123
66934,432,5507,1.5,-2.499512,xXx (2002),"[Action, Crime, Thriller]",2002,-1.531068,-1.612745,-1.945669,-1.985357
74191,474,4012,2.0,-1.683771,Punchline (1988),"[Comedy, Drama]",1988,-1.164015,-1.298387,-1.587484,-1.628389
80857,510,497,1.5,-1.202008,Much Ado About Nothing (1993),"[Comedy, Romance]",1993,-2.101207,-2.160714,-1.445704,-1.447990


In [133]:
# 추가 내용 mean median 비교
opt={
    "mean":user_rating_mean,
    "median":user_rating_median
}

In [134]:
def get_rating_item_user_matrix(movie_ratings_df, rate_type="rating"):
    item_user_matrix = movie_ratings_df.pivot_table(values=rate_type, index='movieId', columns='userId') # 영화별 유저의 weighted mean centering 평점 matrix
    return item_user_matrix.fillna(0)
    
    
def get_genre_item_user_matrix(movie_ratings_df, rating_metric="mean"):
    user_genre_avg = improved_calc_genre_mean(movie_ratings_df) if rating_metric=="mean" else improved_calc_genre_median(movie_ratings_df)
    
    movie_genre_matrix = pd.DataFrame(0, index=movieid_idx, columns=genres_list)
    for genre in genres_list:
        movie_genre_matrix[genre] = movies_df.set_index('movieId')['genres'].apply(lambda x : 1 if genre in x else 0) # apply 적용을 위해 movies_df 의 인덱스 movieId 로 설정
    genre_count = movie_genre_matrix.sum(axis=1)
    movie_genre_matrix = movie_genre_matrix.div(genre_count.replace(0,1), axis=0)
    
    pred_ratings_matrix = user_genre_avg.fillna(0).dot(movie_genre_matrix.T)
    return pred_ratings_matrix.T
    
def get_year_item_user_matrix(movie_ratings_df, rating_metric="mean"):
    user_year_matrix = movie_ratings_df.groupby(['userId','year'])['rating'].mean().unstack() if rating_metric=="mean" else movie_ratings_df.groupby(['userId','year'])['rating'].median().unstack()
    
    temp_df = movies_df.set_index('movieId').loc[movieid_idx]
    movies_year = temp_df['year'].str.strip(to_strip="()")
    movie_year_matrix = pd.get_dummies(movies_year).astype('int64')
    user_year_matrix = user_year_matrix.apply(lambda x: (x-x.mean())/ x.std())
    
    pred_ratings_matrix = user_year_matrix.fillna(0).dot(movie_year_matrix.T)
    return pred_ratings_matrix.T
    

In [135]:
def calc_sim_matrix(rate_type='rating', weights=[1.0, 1.0, 1.0], rating_metric="mean"):
    mat1 = cosine_similarity(get_rating_item_user_matrix(x_train, rate_type=rate_type))
    mat1 = pd.DataFrame(mat1, index=movieid_idx, columns=movieid_idx)
    
    mat2 = cosine_similarity(get_genre_item_user_matrix(x_train, rating_metric))
    mat2 = pd.DataFrame(mat2, index=movieid_idx, columns=movieid_idx)

    mat3 = cosine_similarity(get_year_item_user_matrix(x_train, rating_metric))
    mat3 = pd.DataFrame(mat3, index=movieid_idx, columns=movieid_idx)
    
    weighted_sum = (mat1 * weights[0]) + (mat2 * weights[1]) + (mat3 * weights[2])
    total_sim_matrix = weighted_sum / sum(weights) 
    
    return total_sim_matrix

In [136]:
def RMSE(y_true, y_pred):
    return np.sqrt(np.mean( (np.array(y_true) - np.array(y_pred)) **2))

def score(model, item_user_matrix, sim_matrix, is_centered, rating_metric="mean"): #추가 rating_metric
    id_pairs = zip(x_test['userId'], x_test['movieId']) # x_test : movie_ratings_df
    y_pred = np.array([model(user, movie, item_user_matrix, sim_matrix, is_centered, opt[rating_metric]) for (user, movie) in id_pairs])
    y_true = np.array(x_test['rating'])
    return RMSE(y_true, y_pred)

In [137]:
def ib_cf(user_id, movie_id, item_user_matrix, sim_matrix, is_centered, rating_metric): # model # 추가 user_rating
    if movie_id in sim_matrix:
        sim_scores = sim_matrix[movie_id]
        user_rating = item_user_matrix[user_id]
        non_rating_idx = user_rating[user_rating.isnull()].index
        
        user_rating = user_rating.dropna()
        sim_scores = sim_scores.drop(index=non_rating_idx)
        if is_centered:
            sim_sum = sim_scores.sum()
            if sim_sum==0:
                deviation=0.0
            else:
                deviation = np.dot(sim_scores, user_rating) / sim_sum
            mean = rating_metric[user_id]
            mean_rating = mean+deviation
        else:
            mean_rating = np.dot(sim_scores, user_rating) / sim_scores.sum()
    else:
        mean_rating=3.0
    return mean_rating

In [153]:
og_sim_matrix = calc_sim_matrix("rating")
weighted_sim_matrix = calc_sim_matrix("weighted_centered_rating")
centered_sim_matrix = calc_sim_matrix("centered_rating")

og_item_user_matrix = x_train.pivot_table(values='rating', index='movieId', columns='userId')
weighted_item_user_matrix = x_train.pivot_table(values='weighted_centered_rating', index='movieId', columns='userId')
centered_item_user_matrix = x_train.pivot_table(values='centered_rating', index='movieId', columns='userId')

In [139]:
score(ib_cf, og_item_user_matrix, og_sim_matrix, is_centered=False)

0.9204300462822624

In [140]:
score(ib_cf, weighted_item_user_matrix, sim_matrix=weighted_sim_matrix, is_centered=True)

0.9218011234095814

In [ ]:
score(ib_cf, centered_item_user_matrix, sim_matrix=centered_sim_matrix, is_centered=True) 

0.9088332225414986

In [163]:
# M1 PRO / unified mem 32GB 기준

import time
st = time.time()
score(ib_cf, centered_item_user_matrix, sim_matrix=centered_sim_matrix, is_centered=True) # 시간 측정
et = time.time()

print(f"{et-st:.2f} 초")

4.77 초


In [ ]:
# ---
# median mean 성능 비교
# (1) 유저별 평균 / 중앙값
# (2) similarity 계산시 (위에서 계산한 genre, year 를 이용해 영화별 점수 대표값 계산) median / mean 으로 계산
# ---

centered_sim_matrix_ = calc_sim_matrix("centered_rating",rating_metric="median")
centered_item_user_matrix_ = x_train.pivot_table(values='centered_rating', index='movieId', columns='userId')

score(ib_cf, centered_item_user_matrix_, sim_matrix=centered_sim_matrix_, is_centered=True, rating_metric="median")

0.9485062876956102

# Grid Search - centered_sim_matrix

In [ ]:
from tqdm import tqdm

In [ ]:
w = np.arange(0 , 1.0 , 0.05)

weight_list = []
for i in w:
    for j in w:
        k = 1.0-i-j
        if k>=0:    
            i = round(i,2)
            j = round(j,2)
            k = round(k,2)
            #weight_list.append([i,j,k])
            weight_list.append([k,j,i])

In [ ]:
loss_list = []
best= np.inf

bar = tqdm(weight_list)
best_weight = [np.NaN, np.NaN, np.NaN]
for weight in bar:
    centered_sim_matrix = calc_sim_matrix("centered_rating", weight)
    sc = score(ib_cf, centered_item_user_matrix, sim_matrix=centered_sim_matrix, is_centered=True)
    loss_list.append(sc)
    if sc < best:
        best = sc
        best_weight = weight
    bar.set_description_str(desc=f"loss : {sc} , weight: {weight} | [best] loss : {best} , weight : {best_weight}")

loss : 0.93746367091481 , weight: [0.0, 0.05, 0.95] | [best] loss : 0.8867300806392217 , weight : [0.65, 0.15, 0.2]: 100%|██████████| 221/221 [26:05<00:00,  7.08s/it]  


In [ ]:
loss_list = []
best= np.inf

bar = tqdm(weight_list)
best_weight = [np.NaN, np.NaN, np.NaN]
for weight in bar:
    weighted_sim_matrix = calc_sim_matrix("weighted_centered_rating", weight)
    sc = score(ib_cf, weighted_item_user_matrix, sim_matrix=weighted_sim_matrix, is_centered=True)
    loss_list.append(sc)
    if sc < best:
        best = sc
        best_weight = weight
    bar.set_description_str(desc=f"loss : {sc} , weight: {weight} | [best] loss : {best} , weight : {best_weight}")

loss : 0.938015252410378 , weight: [0.0, 0.05, 0.95] | [best] loss : 0.913655962114638 , weight : [0.7, 0.1, 0.2]: 100%|██████████| 221/221 [24:38<00:00,  6.69s/it]   


In [ ]:
best

0.8867300806392217

In [ ]:
def recommender_system(user_id, item_user_matrix, sim_matrix, is_centered, n_item=10):
    predictions = []
    non_rating_movie = item_user_matrix[user_id][item_user_matrix[user_id].isnull()].index # filtering non rating movieid
    for item in non_rating_movie:
        predictions.append(ib_cf(user_id, item, item_user_matrix, sim_matrix, is_centered))
    
    recom = pd.Series(data=predictions, index=non_rating_movie, dtype=float)
    recom = recom.sort_values(ascending=False)[:n_item]
    return recom


In [ ]:
recommender_system(1, centered_item_user_matrix, centered_sim_matrix, is_centered=True)

movieId
7243     5.471568
7479     5.343484
25898    5.267539
1940     5.121832
8338     5.121832
7065     5.082288
6982     5.068597
959      5.047891
44931    5.040880
4046     5.019100
dtype: float64